# 02 – Feature Engineering

This notebook:
1. Loads raw OHLCV data collected in notebook 01
2. Computes technical indicators (MA, RSI, MACD, Bollinger Bands)
3. Merges macro / sentiment data
4. Handles missing values
5. Performs the train / validation / test split
6. Saves the processed datasets

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, TICKER
from src.data.feature_engineering import FeatureEngineer
from src.data.preprocessor import DataPreprocessor
from src.visualization.plotter import Plotter
from src.utils.data_loader import save_processed_data

In [ ]:
# ── 1. Load raw stock data ──────────────────────────────────────────────────
raw_path = RAW_DATA_DIR / f"{TICKER.replace('^', '')}_ohlcv.csv"
df_raw = pd.read_csv(raw_path, index_col=0, parse_dates=True)
df_raw.index.name = 'Date'
print(f'Raw data shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# ── 2. Technical indicators ─────────────────────────────────────────────────
fe = FeatureEngineer(price_col='Close')
df_features = fe.engineer_all(df_raw, save=False)
print(f'Features shape: {df_features.shape}')
print('Columns:', list(df_features.columns))

In [ ]:
# ── 3. Load macro data (if available) ───────────────────────────────────────
macro_path = RAW_DATA_DIR.parent / 'external' / 'macro_data.csv'
macro_df = None
if macro_path.exists():
    macro_df = pd.read_csv(macro_path, index_col=0, parse_dates=True)
    print(f'Macro data shape: {macro_df.shape}')
else:
    print('Macro data not found – skipping.')

In [ ]:
# ── 4. Merge & handle missing values ────────────────────────────────────────
pre = DataPreprocessor()
df_merged = pre.merge_datasets(df_features, macro_df=macro_df)
df_clean = DataPreprocessor.handle_missing_values(df_merged)
print(f'Merged & cleaned shape: {df_clean.shape}')
print(f'Remaining NaNs: {df_clean.isna().sum().sum()}')

In [ ]:
# ── 5. Correlation heatmap ──────────────────────────────────────────────────
plotter = Plotter()
plotter.plot_correlation_heatmap(df_clean, filename='correlation_heatmap.png')
print('Correlation heatmap saved.')

In [ ]:
# ── 6. Technical indicators chart ───────────────────────────────────────────
plotter.plot_technical_indicators(df_clean, filename='technical_indicators.png')
print('Technical indicators chart saved.')

In [ ]:
# ── 7. Train / validation / test split ──────────────────────────────────────
train, val, test = pre.split_data(df_clean)
print(f'Train : {train.index.min().date()} → {train.index.max().date()}  ({len(train)} days)')
print(f'Val   : {val.index.min().date()} → {val.index.max().date()}  ({len(val)} days)')
print(f'Test  : {test.index.min().date()} → {test.index.max().date()}  ({len(test)} days)')

In [ ]:
# ── 8. Save processed datasets ──────────────────────────────────────────────
save_processed_data(df_clean, 'full_dataset')
save_processed_data(train,    'train')
save_processed_data(val,      'val')
save_processed_data(test,     'test')
print('All datasets saved to data/processed/')

## Summary

Technical indicators computed:
- **Moving Averages**: MA5, MA10, MA20
- **RSI** (14-period)
- **MACD** (12/26/9)
- **Bollinger Bands** (20-period, 2σ)
- **Lag features**: lag_1 … lag_5 (daily returns)
- **Volume**: Volume_MA5, Volume_MA20, OBV

Continue to **03_model_baseline.ipynb** for ARIMA, Prophet, and XGBoost models.